# 08 — Hai hệ số phạt độc lập của QAOA

Sau 90 lượt khảo sát P chung đã review, tách trọng số gán task và phương trình tài nguyên. Giữ p=1, Aer CPU, 1024 shots và maxiter=35. Đây là pilot có giới hạn, chưa phải thí nghiệm xác nhận hay khảo sát độ sâu.

\[
E=\sum_h y_h+\lambda_{\mathrm{assign}}\sum_i(\sum_h x_{i,h}-1)^2
+\lambda_{\mathrm{capacity}}\sum_{h,r}(\sum_i d_{i,r}x_{i,h}+s_{h,r}-C_{h,r}y_h)^2.
\]

Mỗi task vẫn nằm nguyên vẹn trên đúng một host. Slack là tài nguyên dư, không phải phần task. Số biến/qubit giữ nguyên: choice 20, reference 23. Khởi tạo [0.3,0.7] và [0.3,0.05] theo thứ tự tham số hiện tại [beta[0],gamma[0]]. Restart Kernel sau khi chép bộ mã mới, rồi chạy từng cell theo thứ tự.

## Bước 1 — Môi trường và module

Chọn Conda nt409-khiem / Python 3.12 trong VS Code. Cell tìm thư mục repo từ vị trí notebook. Mã nguồn và phiên bản runtime được ghi để tiếp tục đợt chạy an toàn.

In [11]:
import gc
import json
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
             if (p / "src/independent_campaign.py").is_file()
             and (p / "data/instances").is_dir()), None)
assert ROOT is not None, "Chép đủ file src và mở notebook bên trong repo."
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.qubo import build_qubo, energy_parts
from src.ising import build_ising, ising_energy
from src.qaoa_experiment import (
    prepare_reference, probe_qaoa_resources, run_qaoa_experiment,
    save_experiment_report, experiment_environment,
)
from src.penalty_analysis import penalty_diagnostics
from src.independent_campaign import (
    make_design, make_trials, new_campaign, save_campaign, report_row, verify_campaign,
)
assert sys.version_info[:2] == (3, 12), "Chọn kernel Python 3.12 của dự án."
print("Repo:", ROOT)
print("Runtime:", experiment_environment()["runtime"])
print("Nguồn:", experiment_environment()["source_sha256"])

Repo: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Runtime: {'python_version': '3.12.14', 'platform': 'Windows-11-10.0.26200-SP0', 'numpy_version': '2.5.3', 'scipy_version': '1.18.1', 'qiskit_version': '2.5.2', 'qiskit_optimization_version': '0.7.0', 'qiskit_aer_version': '0.17.2'}
Nguồn: {'placement.py': '7cfa397beb22fdf3feddedc0d889e104a619d6ef3e641ee72a62d82b087deb35', 'qubo.py': '7ac7d64500ce1fe73b1b0a34402d51eb88be33372c3906db4b7197552371e9ae', 'ising.py': '7474b47280084059b259b9fadf2971ef26e97e87a53cff14b9b605a2ee2f8b4b', 'qaoa_experiment.py': 'c02713045e03fb80e1185306e9df527359425033a61f462a10a354f30c674f0c', 'penalty_analysis.py': '1ee33001b1ac723df6d2c9bb541ecac27221e30fa77c1dc9fb9777d3d87363ba', 'independent_campaign.py': 'af953f6f78b4fac9ff49522f3cb8f9677499eb3048401219bd7d411f7ac0c1eb'}


## Bước 2 — Ma trận pilot gồm 24 lượt

Hai instance × sáu cặp hệ số × hai khởi tạo × một seed = 24 lượt. (1,1) và (3,3) là đối chứng P chung; bốn cặp còn lại thay riêng một trọng số so với (3,3). Đối chứng được chạy lại trong cùng phiên bản mã mới để kiểm tra trực tiếp. Cùng seed/cấu hình là tái lập, không phải lần lặp độc lập bổ sung cho 90 lượt cũ.

Một seed dùng khám phá. Sau review mới chọn 1–2 ứng viên và kiểm tra seed mới 45–49. Không tăng p, shots hoặc maxiter trong đợt này.

In [12]:
RUN_INSTANCE_IDS = ["p1_choice", "p0_reference"]
PENALTY_PAIRS = {
    "control_P1": [1, 1], "control_P3": [3, 3],
    "assignment_12": [12, 3], "assignment_30": [30, 3],
    "capacity_6": [3, 6], "capacity_12": [3, 12],
}
INITIAL_POINTS = {"goc_cu": [0.3, 0.7], "gamma_nho": [0.3, 0.05]}
SEEDS = [42]
BASE_CONFIG = {
    "shots": 1024, "reps": 1, "maxiter": 35,
    "max_qubits": 23, "max_seconds": 300,
    "aer_threads": 4, "aer_memory_mb": 2048, "min_available_gib": 2.0,
}
RESUME_MANIFEST = None
# Ví dụ tiếp tục: "data/reference_results/phase4b_independent_runs/campaign_....json"
assert RUN_INSTANCE_IDS and len(set(RUN_INSTANCE_IDS)) == len(RUN_INSTANCE_IDS)
assert set(RUN_INSTANCE_IDS) <= {"p1_choice", "p0_reference", "p1_tight", "p1_infeasible"}
instances = {name: json.loads((ROOT / "data/instances" / f"{name}.json").read_text(encoding="utf-8"))
             for name in RUN_INSTANCE_IDS}
DESIGN = make_design(instances, PENALTY_PAIRS, INITIAL_POINTS, SEEDS, BASE_CONFIG)
planned = make_trials(DESIGN)
bundles = {(name, pair_id): build_qubo(item, assignment_penalty=weights[0], capacity_penalty=weights[1])
           for name, item in instances.items() for pair_id, weights in DESIGN["penalty_pairs"].items()}
references = {key: prepare_reference(bundle) for key, bundle in bundles.items()}
print("Số lượt dự kiến:", len(planned))
display(pd.DataFrame([{k: t[k] for k in ["instance_id", "pair_id", "initial_point_id", "seed"]}
                      for t in planned]))

Số lượt dự kiến: 24


,instance_id,pair_id,initial_point_id,seed
0,p1_choice,control_P1,goc_cu,42
1,p1_choice,control_P1,gamma_nho,42
2,p1_choice,control_P3,goc_cu,42
3,p1_choice,control_P3,gamma_nho,42
4,p1_choice,assignment_12,goc_cu,42
5,p1_choice,assignment_12,gamma_nho,42
6,p1_choice,assignment_30,goc_cu,42
7,p1_choice,assignment_30,gamma_nho,42
8,p1_choice,capacity_6,goc_cu,42
9,p1_choice,capacity_6,gamma_nho,42


## Bước 3 — Kiểm chứng trước khi chạy QAOA

Đối chiếu công thức, QUBO và Ising trên mẫu bit cố định/ngẫu nhiên và một cực tiểu tham chiếu ở mỗi mô hình. Kiểm tra hệ số QUBO của hai đối chứng giống bản P chung. Với hai host và dữ liệu nguyên, min(lambda_assign, lambda_capacity)>2 là điều kiện đủ để cực tiểu toàn cục hợp lệ nếu bài toán có nghiệm. (1,1) được đánh giá bằng tham chiếu riêng; không suy ra bảo đảm chung từ P=3.

In [3]:
rng = np.random.default_rng(409)
model_rows = []
checked = 0
for (name, pair_id), bundle in bundles.items():
    ref = references[name, pair_id]
    operator, offset = build_ising(bundle)
    vectors = [[0] * operator.num_qubits, ref["example_minimizer_bits"],
               *rng.integers(0, 2, size=(8, operator.num_qubits)).tolist()]
    for bits in vectors:
        formula = energy_parts(bundle, bits)["energy"]
        assert np.isclose(formula, bundle["qubo"].objective.evaluate(bits), rtol=0, atol=1e-7)
        assert np.isclose(formula, ising_energy(operator, offset, bits), rtol=0, atol=1e-7)
        checked += 1
    if bundle["penalty"] is not None:
        old = build_qubo(instances[name], penalty=bundle["penalty"])
        assert old["qubo"].objective.constant == bundle["qubo"].objective.constant
        np.testing.assert_array_equal(old["qubo"].objective.linear.to_array(), bundle["qubo"].objective.linear.to_array())
        np.testing.assert_array_equal(old["qubo"].objective.quadratic.to_array(), bundle["qubo"].objective.quadratic.to_array())
    if ref["instance_feasible"]:
        assert ref["all_qubo_minimizers_feasible"], "Cặp hệ số này có cực tiểu QUBO vi phạm; cần review trước."
    model_rows.append({"instance": name, "pair": pair_id,
        "lambda_assign": bundle["penalties"]["assignment"], "lambda_capacity": bundle["penalties"]["capacity"],
        "qubit": operator.num_qubits, "statevector_MiB": 16 * 2**operator.num_qubits / 2**20,
        "original_optimum": ref["original_best_objective"], "qubo_minimum": ref["minimum_energy"],
        "all_minima_feasible": ref["all_qubo_minimizers_feasible"],
        "uniform_feasible_pct": 100*ref["uniform_baseline"]["feasible_probability"]})
print("PASS — công thức/QUBO/Ising và đối chứng P chung; số vector:", checked)
display(pd.DataFrame(model_rows))

PASS — công thức/QUBO/Ising và đối chứng P chung; số vector: 120


,instance,pair,lambda_assign,lambda_capacity,qubit,statevector_MiB,original_optimum,qubo_minimum,all_minima_feasible,uniform_feasible_pct
0,p1_choice,control_P1,1.0,1.0,20,16.0,1,1.0,True,1.171875
1,p1_choice,control_P3,3.0,3.0,20,16.0,1,1.0,True,1.171875
2,p1_choice,assignment_12,12.0,3.0,20,16.0,1,1.0,True,1.171875
3,p1_choice,assignment_30,30.0,3.0,20,16.0,1,1.0,True,1.171875
4,p1_choice,capacity_6,3.0,6.0,20,16.0,1,1.0,True,1.171875
5,p1_choice,capacity_12,3.0,12.0,20,16.0,1,1.0,True,1.171875
6,p0_reference,control_P1,1.0,1.0,23,128.0,2,2.0,True,0.097656
7,p0_reference,control_P3,3.0,3.0,23,128.0,2,2.0,True,0.097656
8,p0_reference,assignment_12,12.0,3.0,23,128.0,2,2.0,True,0.097656
9,p0_reference,assignment_30,30.0,3.0,23,128.0,2,2.0,True,0.097656


## Bước 4 — Đo thử tài nguyên

Mỗi mô hình lấy mẫu một lần trước khi tối ưu. Ước tính thời gian chỉ gồm phần lấy mẫu, không phải cam kết deadline. Aer dùng một statevector tại một thời điểm; max_memory_mb giới hạn bộ nhớ trạng thái lượng tử, không giới hạn toàn bộ Python. RSS đỉnh gồm hoạt động kernel trước đó.

In [4]:
probes = {}
for key, bundle in bundles.items():
    trial = next(t for t in planned if (t["instance_id"], t["pair_id"]) == key)
    print("Đo thử:", *key, flush=True)
    probes[key] = probe_qaoa_resources(bundle, trial["config"])
display(pd.DataFrame([{"instance": key[0], "pair": key[1],
    "sampling_seconds": value["sampling_seconds"],
    "rough_sampling_seconds_per_run": value["rough_maxiter_plus_final_sampling_seconds"],
    "process_peak_rss_mib": value["resources"]["process_peak_rss_mib"]}
    for key, value in probes.items()]))
rough = sum(probes[t["instance_id"], t["pair_id"]]["rough_maxiter_plus_final_sampling_seconds"] for t in planned)
print("Ước tính riêng phần lấy mẫu cả ma trận (phút):", round(rough / 60, 2))
if any(v["rough_maxiter_plus_final_sampling_seconds"] >= BASE_CONFIG["max_seconds"] for v in probes.values()):
    raise RuntimeError("Đo thử chạm ngân sách. Xem tài nguyên trước khi chạy Bước 6.")

Đo thử: p1_choice control_P1
Đo thử: p1_choice control_P3
Đo thử: p1_choice assignment_12
Đo thử: p1_choice assignment_30
Đo thử: p1_choice capacity_6
Đo thử: p1_choice capacity_12
Đo thử: p0_reference control_P1
Đo thử: p0_reference control_P3
Đo thử: p0_reference assignment_12
Đo thử: p0_reference assignment_30
Đo thử: p0_reference capacity_6
Đo thử: p0_reference capacity_12


,instance,pair,sampling_seconds,rough_sampling_seconds_per_run,process_peak_rss_mib
0,p1_choice,control_P1,0.071393,2.570166,185.449219
1,p1_choice,control_P3,0.064962,2.338650,186.328125
2,p1_choice,assignment_12,0.059165,2.129958,186.871094
3,p1_choice,assignment_30,0.058876,2.119550,187.078125
4,p1_choice,capacity_6,0.056389,2.030004,187.230469
5,p1_choice,capacity_12,0.055508,1.998292,187.230469
6,p0_reference,control_P1,0.480632,17.302745,299.531250
7,p0_reference,control_P3,0.482633,17.374802,299.785156
8,p0_reference,assignment_12,0.492904,17.744548,299.855469
9,p0_reference,assignment_30,0.481413,17.330861,299.894531


Ước tính riêng phần lấy mẫu cả ma trận (phút): 3.94


## Bước 5 — Tạo hoặc tiếp tục manifest

Manifest ghi cả thiết kế, nguồn mã, runtime, input và danh sách lượt. Khi tiếp tục, giữ nguyên thiết kế và chọn đúng manifest notebook 08. Mã đọc một báo cáo tại một thời điểm và kiểm tra lại mẫu thô. Tệp nguồn hoặc runtime thay đổi thì cần một đợt mới.

In [5]:
if RESUME_MANIFEST is None:
    campaign = new_campaign(DESIGN)
    manifest_path = ROOT / "data/reference_results/phase4b_independent_runs" / f"campaign_{campaign['campaign_id']}.json"
    save_campaign(manifest_path, campaign)
else:
    manifest_path = (ROOT / RESUME_MANIFEST).resolve()
    assert manifest_path.is_relative_to(ROOT), "Manifest phải nằm trong repo."
    campaign = json.loads(manifest_path.read_text(encoding="utf-8"))
    verify_campaign(ROOT, campaign, DESIGN, bundles, references)
print("Manifest:", manifest_path.relative_to(ROOT).as_posix())
print("Đã hoàn tất / dự kiến:", campaign["completed_run_count"], "/", campaign["planned_run_count"])

Manifest: data/reference_results/phase4b_independent_runs/campaign_20261006T080102274154Z.json
Đã hoàn tất / dự kiến: 0 / 24


## Bước 6 — Chạy pilot và lưu ngay từng lượt

Một lượt hoàn tất có 0 nghiệm vẫn được lưu. Nếu bị ngắt hoặc vượt ngân sách, dừng đợt và giữ các lượt hoàn tất trước đó; lượt chưa hoàn tất không được ghi thành kết quả. Chỉ mục được thay thế nguyên tử sau mỗi lượt để hỗ trợ resume.

In [6]:
completed = {row["run_key"] for row in campaign["completed_runs"]}
for trial in planned:
    if trial["run_key"] in completed:
        print("Bỏ qua lượt đã hoàn tất:", trial["instance_id"], trial["pair_id"], trial["initial_point_id"], trial["seed"])
        continue
    assert experiment_environment() == DESIGN["environment"], "Mã/runtime thay đổi. Tạo đợt mới sau Restart Kernel."
    key = trial["instance_id"], trial["pair_id"]
    print("START", *key, trial["initial_point_id"], "seed", trial["seed"], flush=True)
    report = run_qaoa_experiment(bundles[key], trial["config"], references[key])
    if report["status"] != "completed":
        print("Dừng đợt:", report["stop_reason"], "Lượt chưa hoàn tất không được lưu.")
        del report
        gc.collect()
        break
    assert report["environment"] == DESIGN["environment"]
    report["experiment"] = {"campaign_id": campaign["campaign_id"], "design_hash": campaign["design_hash"],
        "run_key": trial["run_key"], "pair_id": trial["pair_id"],
        "initial_point_id": trial["initial_point_id"], "purpose": "controlled_independent_penalty_pilot"}
    started = perf_counter()
    report["penalty_analysis"] = penalty_diagnostics(bundles[key], report)
    report["penalty_analysis_seconds"] = perf_counter() - started
    path = save_experiment_report(ROOT, report)
    row = report_row(ROOT, path, report, trial)
    campaign["completed_runs"].append(row)
    save_campaign(manifest_path, campaign)
    completed.add(trial["run_key"])
    print("DONE | hợp lệ:", round(100*row["feasible_probability"], 4), "% | host tốt nhất:",
          row["observed_best_objective"], "| thời gian:", round(row["elapsed_seconds"], 2), "s", flush=True)
    del report
    gc.collect()

START p1_choice control_P1 goc_cu seed 42
  evaluation 1: mean E=60.133789, elapsed=0.1s
  evaluation 5: mean E=58.869141, elapsed=0.4s
  evaluation 10: mean E=62.505859, elapsed=0.7s
  evaluation 15: mean E=36.001953, elapsed=1.1s
  evaluation 20: mean E=31.137695, elapsed=1.4s
  evaluation 25: mean E=29.155273, elapsed=1.8s
  evaluation 30: mean E=27.970703, elapsed=2.1s
DONE | hợp lệ: 1.9531 % | host tốt nhất: 2 | thời gian: 2.28 s
START p1_choice control_P1 gamma_nho seed 42
  evaluation 1: mean E=110.847656, elapsed=0.1s
  evaluation 5: mean E=55.814453, elapsed=0.4s
  evaluation 10: mean E=55.719727, elapsed=0.7s
  evaluation 15: mean E=53.375977, elapsed=1.0s
  evaluation 20: mean E=53.169922, elapsed=1.4s
  evaluation 25: mean E=53.853516, elapsed=1.7s
DONE | hợp lệ: 2.3438 % | host tốt nhất: 2 | thời gian: 1.98 s
START p1_choice control_P3 goc_cu seed 42
  evaluation 1: mean E=159.282227, elapsed=0.1s
  evaluation 5: mean E=175.377930, elapsed=0.4s
  evaluation 10: mean E=173.

## Bước 7 — Xem từng lượt

one_hot_probability là xác suất tất cả task được gán đúng một host, chưa xét dung lượng. feasible_probability xét cả gán và dung lượng; optimal_placement_probability còn yêu cầu số host nhỏ nhất. gap=0 chỉ cho phương án hợp lệ đạt mục tiêu tối ưu. None/NaN là thiếu nghiệm hoặc mục tiêu không tồn tại, không phải gap=0. Đồng hạng không có unique mode.

In [7]:
results = pd.DataFrame(campaign["completed_runs"])
if results.empty:
    print("Chưa có lượt hoàn tất.")
else:
    table = results[["instance_id", "pair_id", "initial_point_id", "seed", "assignment_penalty", "capacity_penalty",
        "one_hot_probability", "feasible_probability", "optimal_placement_probability",
        "observed_best_objective", "optimality_gap", "dominant_bitstring_count", "unique_mode_valid", "elapsed_seconds"]].copy()
    for key in ["one_hot_probability", "feasible_probability", "optimal_placement_probability"]:
        table[key] = 100 * pd.to_numeric(table[key])
    display(table.round(6))
    print("Ba cột probability ở bảng này đã đổi sang phần trăm.")

,instance_id,pair_id,initial_point_id,seed,assignment_penalty,capacity_penalty,one_hot_probability,feasible_probability,optimal_placement_probability,observed_best_objective,optimality_gap,dominant_bitstring_count,unique_mode_valid,elapsed_seconds
0,p1_choice,control_P1,goc_cu,42,1.0,1.0,4.882812,1.953125,0.000000,2.0,1.0,3,None,2.279237
1,p1_choice,control_P1,gamma_nho,42,1.0,1.0,8.984375,2.343750,0.000000,2.0,1.0,2,None,1.982066
2,p1_choice,control_P3,goc_cu,42,3.0,3.0,9.472656,3.222656,0.195312,1.0,0.0,3,None,1.991736
3,p1_choice,control_P3,gamma_nho,42,3.0,3.0,6.250000,2.539062,0.097656,1.0,0.0,1,False,2.155919
4,p1_choice,assignment_12,goc_cu,42,12.0,3.0,8.593750,2.539062,0.000000,2.0,1.0,9,None,1.978289
5,p1_choice,assignment_12,gamma_nho,42,12.0,3.0,5.566406,2.343750,0.097656,1.0,0.0,1,False,2.109299
6,p1_choice,assignment_30,goc_cu,42,30.0,3.0,11.718750,2.343750,0.097656,1.0,0.0,2,None,1.880034
7,p1_choice,assignment_30,gamma_nho,42,30.0,3.0,6.835938,2.441406,0.097656,1.0,0.0,3,None,2.095970
8,p1_choice,capacity_6,goc_cu,42,3.0,6.0,7.031250,2.343750,0.097656,1.0,0.0,2,None,2.097627
9,p1_choice,capacity_6,gamma_nho,42,3.0,6.0,7.031250,2.636719,0.097656,1.0,0.0,2,None,2.383861


Ba cột probability ở bảng này đã đổi sang phần trăm.


## Bước 8 — So sánh từng cặp và khởi tạo

Chọn ứng viên từ xác suất hợp lệ/tối ưu, tỷ lệ lượt thành công và chi phí. Năng lượng thô có thang khác giữa các cặp nên không dùng để xếp hạng chúng. Sai số tài nguyên gồm slack sai, không chỉ vượt CPU/RAM thật. Với một seed, độ lệch chuẩn giữa seed chưa xác định; kết quả chưa chứng minh một cặp thắng ổn định.

In [8]:
summary = pd.DataFrame()
if not results.empty:
    def optional_sum(series):
        return None if series.isna().all() else int(series.sum())
    summary = results.groupby(["instance_id", "pair_id", "assignment_penalty", "capacity_penalty", "initial_point_id"], sort=False).agg(
        completed_runs=("run_key", "size"), valid_runs=("found_valid", "sum"),
        optimal_runs=("found_optimal", optional_sum),
        onehot_pct_mean=("one_hot_probability", lambda s: 100*s.mean()),
        feasible_pct_mean=("feasible_probability", lambda s: 100*s.mean()),
        feasible_pct_std=("feasible_probability", lambda s: 100*s.std(ddof=1)),
        optimal_pct_mean=("optimal_placement_probability", lambda s: 100*pd.to_numeric(s).mean()),
        mean_assignment_error=("mean_assignment_error", "mean"),
        mean_capacity_equation_error=("mean_capacity_equation_error", "mean"),
        mean_solver_seconds=("elapsed_seconds", "mean"),
    ).reset_index()
    summary["uniform_feasible_pct"] = summary.apply(
        lambda row: 100*references[row["instance_id"], row["pair_id"]]["uniform_baseline"]["feasible_probability"], axis=1)
    display(summary.round(6))
    print("Một seed: *_std là NaN. Không dùng số liệu này để tuyên bố cấu hình thắng.")

,instance_id,pair_id,assignment_penalty,capacity_penalty,initial_point_id,completed_runs,valid_runs,optimal_runs,onehot_pct_mean,feasible_pct_mean,feasible_pct_std,optimal_pct_mean,mean_assignment_error,mean_capacity_equation_error,mean_solver_seconds,uniform_feasible_pct
0,p1_choice,control_P1,1.0,1.0,goc_cu,1,1,0,4.882812,1.953125,NaN,0.000000,2.169922,23.894531,2.279237,1.171875
1,p1_choice,control_P1,1.0,1.0,gamma_nho,1,1,0,8.984375,2.343750,NaN,0.000000,1.989258,48.809570,1.982066,1.171875
2,p1_choice,control_P3,3.0,3.0,goc_cu,1,1,1,9.472656,3.222656,NaN,0.195312,2.023438,44.545898,1.991736,1.171875
3,p1_choice,control_P3,3.0,3.0,gamma_nho,1,1,1,6.250000,2.539062,NaN,0.097656,2.002930,38.707031,2.155919,1.171875
4,p1_choice,assignment_12,12.0,3.0,goc_cu,1,1,0,8.593750,2.539062,NaN,0.000000,1.998047,41.616211,1.978289,1.171875
5,p1_choice,assignment_12,12.0,3.0,gamma_nho,1,1,1,5.566406,2.343750,NaN,0.097656,1.995117,35.292969,2.109299,1.171875
6,p1_choice,assignment_30,30.0,3.0,goc_cu,1,1,1,11.718750,2.343750,NaN,0.097656,1.902344,44.507812,1.880034,1.171875
7,p1_choice,assignment_30,30.0,3.0,gamma_nho,1,1,1,6.835938,2.441406,NaN,0.097656,1.943359,34.947266,2.095970,1.171875
8,p1_choice,capacity_6,3.0,6.0,goc_cu,1,1,1,7.031250,2.343750,NaN,0.097656,2.062500,45.244141,2.097627,1.171875
9,p1_choice,capacity_6,3.0,6.0,gamma_nho,1,1,1,7.031250,2.636719,NaN,0.097656,1.990234,34.672852,2.383861,1.171875


Một seed: *_std là NaN. Không dùng số liệu này để tuyên bố cấu hình thắng.


## Bước 9 — Xem mẫu đo của một lượt

Tần suất cao nhất và nghiệm hợp lệ tốt nhất là hai quy tắc đầu ra khác nhau. Chương trình không sửa task/slack trước khi chấm. Chọn INSPECT_RUN_INDEX để xem một lượt khác.

In [9]:
INSPECT_RUN_INDEX = 0
if campaign["completed_runs"]:
    entry = campaign["completed_runs"][INSPECT_RUN_INDEX]
    inspected = json.loads((ROOT / entry["report"]).read_text(encoding="utf-8"))
    print("Nguồn:", entry["report"])
    print("Hai hệ số:", inspected["penalties"], "| thứ tự tham số:", inspected["parameter_names"])
    print("Nghiệm hợp lệ tốt nhất:", inspected["metrics"]["best_observed_valid"])
    print("Chẩn đoán:", inspected["penalty_analysis"])
    display(pd.DataFrame([{"probability": s["probability"], "energy": s["energy"],
        "feasible": s["decoded"]["feasible"], "hosts": s["decoded"]["objective"],
        "x": s["decoded"]["x"], "y": s["decoded"]["y"]} for s in inspected["final_samples"][:5]]))
    del inspected
    gc.collect()

Nguồn: data/reference_results/phase4b_independent_runs/p1_choice_a1_c1_p1_seed42_20261006T080224967166Z.json
Hai hệ số: {'assignment': 1.0, 'capacity': 1.0} | thứ tự tham số: ['β[0]', 'γ[0]']
Nghiệm hợp lệ tốt nhất: {'bits': [0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0], 'energy': 5.0, 'probability': 0.0009765625, 'feasible': True, 'objective': 2, 'one_hot_ok': True, 'capacity_ok': True, 'x': [[0, 1], [1, 0], [1, 0], [1, 0]], 'y': [1, 1], 'usage': [[4, 4], [1, 1]]}
Chẩn đoán: {'mean_active_hosts': 1.6064453125, 'mean_assignment_error': 2.169921875, 'mean_capacity_equation_error': 23.89453125, 'mean_total_squared_error': 26.064453125, 'reconstructed_mean_energy': 27.6708984375, 'dominant_bitstring_count': 3, 'dominant_valid_bitstring_count': 0, 'unique_mode_valid': None, 'one_hot_probability': 0.048828125}


,probability,energy,feasible,hosts,x,y
0,0.001953,26.0,False,None,"[[0, 0], [0, 0], [0, 1], [0, 1]]","[1, 1]"
1,0.001953,22.0,False,None,"[[1, 0], [0, 1], [0, 0], [0, 0]]","[1, 1]"
2,0.001953,22.0,False,None,"[[0, 1], [1, 0], [0, 0], [0, 0]]","[1, 1]"
3,0.000977,18.0,False,None,"[[0, 1], [0, 0], [0, 1], [0, 0]]","[1, 1]"
4,0.000977,9.0,False,None,"[[1, 0], [1, 1], [1, 0], [0, 0]]","[1, 1]"


## Bước 10 — Kiểm chứng và bàn giao kết quả

Đối chiếu lại manifest với thiết kế, input, runtime, nguồn mã, mô hình, mẫu thô, metrics và bảng từng lượt. PASS kiểm tra tính nhất quán dữ liệu, không có nghĩa QAOA luôn tìm được nghiệm hay đạt chất lượng mong muốn.

In [10]:
saved = json.loads(manifest_path.read_text(encoding="utf-8"))
verified = verify_campaign(ROOT, saved, DESIGN, bundles, references)
assert verified == campaign["completed_runs"]
print("PASS — manifest, thiết kế, nguồn mã, mô hình và các mẫu đo khớp.")
print("Hoàn tất / dự kiến:", saved["completed_run_count"], "/", saved["planned_run_count"])
print("Lượt còn lại:", len(saved["remaining_run_keys"]))
print("Gửi để review: notebook có output, manifest và các JSON được manifest tham chiếu.")
print("Thư mục:", manifest_path.parent.relative_to(ROOT).as_posix())

PASS — manifest, thiết kế, nguồn mã, mô hình và các mẫu đo khớp.
Hoàn tất / dự kiến: 24 / 24
Lượt còn lại: 0
Gửi để review: notebook có output, manifest và các JSON được manifest tham chiếu.
Thư mục: data/reference_results/phase4b_independent_runs


## Mốc sau pilot

Lưu notebook có output, manifest và báo cáo nguồn. Giữ nguyên 90 lượt cũ. Gửi đợt này để review trước khi chọn 1–2 ứng viên cho seed mới 45–49. Nếu không có tín hiệu cải thiện, đóng ma trận penalty và chuyển sang độ sâu p=1/2/3 với đối chứng ngân sách.

Để resume: Restart Kernel, giữ cùng thiết kế/mã/runtime/input, đặt RESUME_MANIFEST đến manifest cũ, rồi chạy các cell theo thứ tự. Lượt đã có trong manifest được bỏ qua. Chạy lại Bước 6 trong cùng kernel cũng bỏ qua chúng. Nếu thay seed/cặp/input/mã, đặt RESUME_MANIFEST=None để tạo đợt mới. Đối chứng cùng seed là tái lập; không gộp chúng thành các lần lặp độc lập.

---

## English explanation

This bounded pilot separates assignment and capacity-equation weights while preserving host-count minimization, binary x/y/slack registers and the CPU Aer backend. Default: choice/reference, six pairs, two starts, seed 42, depth 1, shots 1024 and maxiter 35: 24 trials. Pairs (1,1)/(3,3) are fresh common-P controls in the current code version; repeated identical seeds/settings are reproduction controls rather than new independent repetitions. The other pairs are (12,3), (30,3), (3,6), (3,12).

Restart the kernel after installing source files. Run steps 1–10 sequentially: environment, design, exact model checks, resource probes, campaign creation/resume, sequential runs, per-trial tables, grouped comparison, sample inspection and final source verification. Each completed trial, including zero-feasibility outcomes, is saved immediately with schema 3, both weights, model fingerprint, source hashes, runtime and budget. Reports/manifests use data/reference_results/phase4b_independent_runs. The manifest is replaced atomically and resume requires unchanged design/input/code/runtime. Reports from notebooks 06/07 remain intact and cannot serve as notebook 08 resume manifests.

Score only measured bits. Distinguish one-hot probability, original feasibility, original optimality and full-QUBO ground states; disclose frequency ties and the best-valid postselection rule. Capacity-equation error includes wrong slack. Compare quality/success/cost rather than raw energy across different weights. One pilot seed per group does not support a stable winner; seed standard deviation is undefined.

After review, retain at most one or two candidates and validate with controls on fresh seeds provisionally 45–49. If penalty changes do not help, stop widening the grid and test depth 1/2/3 with measured resource costs and matched optimizer-budget controls. This notebook does not perform that later study. Linux reduced-budget checks are software verification, not the user's Windows experimental results.

Sources: [LinearEqualityToPenalty, Qiskit Optimization 0.7](https://qiskit-community.github.io/qiskit-optimization/stubs/qiskit_optimization.converters.LinearEqualityToPenalty.html) and [QuadraticProgram.minimize](https://qiskit-community.github.io/qiskit-optimization/stubs/qiskit_optimization.QuadraticProgram.html#qiskit_optimization.QuadraticProgram.minimize).